# Experiment 2 — EDCR on CIFAR-100 Hierarchical Classifier

**TRAP formula:** `f′(x; g(f(x), θ))` — adaptation via post-hoc symbolic correction of a frozen f.  
**Week 3, Days 1–2**

## Task
1. Train a ResNet-50 classifier on CIFAR-100 (20 superclasses / 100 fine classes)  
2. Apply PyEDCR f-EDR to learn error-detection rules without prior constraints  
3. Report class-level precision/recall **before vs. after** EDCR correction  
4. Print the learned symbolic rules  

**Success criterion:**  
- ≥ 3% macro-F1 lift on at least one class  
- ≥ 5 human-interpretable rules of the form "if predicted fine-label = X and condition C, suspect error"  

## References
- Xi et al. 2025 — EDCR (original), arXiv:2308.14250  
- Kricheli et al. 2024 — f-EDR (hierarchical), CIKM 2024, DOI 10.1145/3627673.3679918  
- PyEDCR: https://github.com/lab-v2/PyEDCR

## 0 · Environment check

In [ ]:
import importlib, sys
for pkg in ['torch', 'torchvision', 'pyedcr', 'sklearn', 'numpy', 'matplotlib', 'pandas']:
    try:
        importlib.import_module(pkg)
        print(f'  ✓  {pkg}')
    except ImportError:
        print(f'  ✗  {pkg}')

import torch
print(f'\nPyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')

## 1 · Imports

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import classification_report, f1_score
from pathlib import Path
from tqdm import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_ROOT = Path('../../data')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
CKPT_PATH = Path('resnet50_cifar100.pt')

# CIFAR-100: 20 superclasses, each containing 5 fine classes
# Superclass → fine-class mapping from the CIFAR-100 paper
CIFAR100_SUPERCLASS_MAP = {
    'aquatic mammals':   [4, 30, 55, 72, 95],
    'fish':              [1, 32, 67, 73, 91],
    'flowers':           [54, 62, 70, 82, 92],
    'food containers':   [9, 10, 16, 28, 61],
    'fruit and vegetables': [0, 51, 53, 57, 83],
    'household electrical devices': [22, 39, 40, 86, 87],
    'household furniture': [5, 20, 25, 84, 94],
    'insects':           [6, 7, 14, 18, 24],
    'large carnivores':  [3, 42, 43, 88, 97],
    'large man-made outdoor things': [12, 17, 37, 68, 76],
    'large natural outdoor scenes':  [23, 33, 49, 60, 71],
    'large omnivores and herbivores': [15, 19, 21, 31, 38],
    'medium-sized mammals': [34, 63, 64, 66, 75],
    'non-insect invertebrates': [26, 45, 77, 79, 99],
    'people':            [2, 11, 35, 46, 98],
    'reptiles':          [27, 29, 44, 78, 93],
    'small mammals':     [36, 50, 65, 74, 80],
    'trees':             [47, 52, 56, 59, 96],
    'vehicles 1':        [8, 13, 48, 58, 90],
    'vehicles 2':        [41, 69, 81, 85, 89],
}
# Invert: fine-class → superclass index
FINE_TO_SUPER = {}
for super_idx, (_, fine_classes) in enumerate(CIFAR100_SUPERCLASS_MAP.items()):
    for fc in fine_classes:
        FINE_TO_SUPER[fc] = super_idx

print(f'Device: {DEVICE}')
print(f'Superclass mapping loaded: {len(CIFAR100_SUPERCLASS_MAP)} superclasses')

## 2 · Load CIFAR-100

In [ ]:
MEAN = (0.5071, 0.4867, 0.4408)
STD  = (0.2675, 0.2565, 0.2761)

train_tfm = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])
test_tfm = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])

train_ds = torchvision.datasets.CIFAR100(DATA_ROOT, train=True,  download=True, transform=train_tfm)
test_ds  = torchvision.datasets.CIFAR100(DATA_ROOT, train=False, download=True, transform=test_tfm)

# Use a calibration split for EDCR rule learning (20% of train)
n_train = int(0.8 * len(train_ds))
n_cal   = len(train_ds) - n_train
train_sub = Subset(train_ds, range(n_train))
cal_sub   = Subset(train_ds, range(n_train, len(train_ds)))

train_loader = DataLoader(train_sub, batch_size=128, shuffle=True,  num_workers=2, pin_memory=True)
cal_loader   = DataLoader(cal_sub,   batch_size=256, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,   batch_size=256, shuffle=False, num_workers=2)

print(f'Train: {len(train_sub)}, Calibration: {n_cal}, Test: {len(test_ds)}')

## 3 · ResNet-50 baseline

In [ ]:
def build_resnet50(num_classes=100):
    model = torchvision.models.resnet50(weights=None)
    # CIFAR-100 is 32×32 — patch stem
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        correct += (logits.argmax(1) == labels).sum().item()
        total += len(labels)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE)
        preds = model(imgs).argmax(1).cpu()
        all_preds.append(preds)
        all_labels.append(labels)
    return torch.cat(all_preds).numpy(), torch.cat(all_labels).numpy()


# Train or load from checkpoint
model = build_resnet50().to(DEVICE)
NUM_EPOCHS = 50  # reduce for quick tests

if CKPT_PATH.exists():
    print(f'Loading checkpoint from {CKPT_PATH}')
    model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
else:
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

    for epoch in tqdm(range(NUM_EPOCHS), desc='Training ResNet-50'):
        loss, acc = train_one_epoch(model, train_loader, optimizer, criterion)
        scheduler.step()
        if (epoch + 1) % 10 == 0:
            print(f'  Epoch {epoch+1:3d}  loss={loss:.4f}  train_acc={acc:.4f}')

    torch.save(model.state_dict(), CKPT_PATH)
    print(f'Saved to {CKPT_PATH}')

preds_test, labels_test = evaluate(model, test_loader)
baseline_f1 = f1_score(labels_test, preds_test, average='macro')
print(f'\nBaseline macro-F1 (before EDCR): {baseline_f1:.4f}')

## 4 · PyEDCR — f-EDR rule learning

f-EDR (Kricheli et al. CIKM 2024) learns rules of the form:
> "if predicted fine-label = X and parent superclass ≠ superclass(X), suspect error"

See PyEDCR docs and the CIKM paper for the full rule language.

**Note:** PyEDCR documentation lags the code — consult `tests/` in the repo to
reconstruct the f-EDR API if needed.

In [ ]:
# Collect calibration predictions for rule learning
preds_cal, labels_cal = evaluate(model, cal_loader)

# Build feature matrix: for EDCR we pass predicted labels + superclass memberships
# (EDCR operates on predicted-label features, not raw image features)
def build_edcr_features(preds, fine_to_super):
    """Return (pred_fine, pred_super) pairs."""
    pred_super = np.array([fine_to_super[p] for p in preds])
    return np.stack([preds, pred_super], axis=1)  # shape (N, 2)

X_cal = build_edcr_features(preds_cal, FINE_TO_SUPER)
y_cal = labels_cal  # true fine labels

print(f'Calibration features shape: {X_cal.shape}')
print(f'Calibration accuracy: {(preds_cal == labels_cal).mean():.4f}')

In [ ]:
# TODO: instantiate and train PyEDCR f-EDR model
#
# The PyEDCR API (from lab-v2/PyEDCR, CIKM-2024 branch) is roughly:
#
# from pyedcr import EDCR, EDCRConfig
#
# config = EDCRConfig(
#     hierarchy=FINE_TO_SUPER,
#     min_support=0.01,
#     min_confidence=0.8,
#     max_rule_length=3,
# )
# edcr = EDCR(config)
# edcr.fit(X_cal, y_cal, preds_cal)
#
# Consult: https://github.com/lab-v2/PyEDCR/tree/main/tests
# and the CIKM 2024 paper §3 for the exact API.

print('TODO: fit PyEDCR model on calibration split')

In [ ]:
# TODO: apply EDCR rules to test predictions
#
# X_test = build_edcr_features(preds_test, FINE_TO_SUPER)
# preds_corrected = edcr.predict(X_test, preds_test)
#
# Placeholder for now:
preds_corrected = preds_test.copy()

after_f1 = f1_score(labels_test, preds_corrected, average='macro')
print(f'Macro-F1 before EDCR: {baseline_f1:.4f}')
print(f'Macro-F1 after  EDCR: {after_f1:.4f}')
print(f'Delta:                {after_f1 - baseline_f1:+.4f}')

## 5 · Learned rules

In [ ]:
# TODO: print learned rules
#
# for rule in edcr.rules_[:10]:
#     print(rule)

# Expected format (from CIKM paper):
# RULE: if pred_fine=42 (bear) AND pred_super≠8 (large carnivores) → error
# RULE: if pred_fine=67 (pike) AND pred_super≠1 (fish) → error

print('TODO: print top-10 learned EDCR rules')

## 6 · Before vs. after comparison

In [ ]:
# Per-class F1 comparison
f1_before = f1_score(labels_test, preds_test,      average=None, labels=range(100))
f1_after  = f1_score(labels_test, preds_corrected, average=None, labels=range(100))
delta     = f1_after - f1_before

# Top 10 classes with largest improvement
top10_idx = np.argsort(delta)[::-1][:10]
print('Top-10 classes by F1 improvement after EDCR:')
for idx in top10_idx:
    print(f'  class {idx:3d}: {f1_before[idx]:.4f} → {f1_after[idx]:.4f}  (Δ={delta[idx]:+.4f})')

# Bar chart
fig, ax = plt.subplots(figsize=(12, 4))
sorted_idx = np.argsort(delta)
colors = ['green' if d > 0 else 'red' for d in delta[sorted_idx]]
ax.bar(range(100), delta[sorted_idx], color=colors, alpha=0.7)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Class (sorted by Δ F1)')
ax.set_ylabel('Δ macro-F1 after EDCR')
ax.set_title('Per-class F1 change after EDCR correction')
plt.tight_layout()
plt.savefig('edcr_f1_delta.png', dpi=150)
plt.show()

## 7 · TRAP annotation

**TRAP formula:** `f′(x; g(f(x), θ))`

- `f` is the frozen ResNet-50 — it maps images to fine-class predictions.  
- `g` is PyEDCR: it takes `(f(x), θ)` — predicted label and hierarchy constraints —
  and returns symbolic correction rules.  
- `f′` is the corrected classifier: original predictions overridden by EDCR rules.  
- The **Adaptation** arm of TRAP is instantiated: the system detects and corrects its own
  errors without retraining f.  
- The learned rules are the **Transparency** artefact: they are human-inspectable symbolic
  objects (not black-box corrections).  

**Observations from experiment:**  
*(fill in after running)*